In [ ]:
import random
import time

# We assume docs and char_kgrams are already defined from Q1

def load_doc(path):
    with open(path, 'r') as f:
        return f.read().strip()

doc_paths = {
    'D1': 'minhash/D1.txt',
    'D2': 'minhash/D2.txt',
    'D3': 'minhash/D3.txt',
    'D4': 'minhash/D4.txt',
}
docs = {name: load_doc(path) for name, path in doc_paths.items()}

def char_kgrams(text, k):
    return set(text[i:i+k] for i in range(len(text) - k + 1))

def jaccard(set_a, set_b):

    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union > 0 else 0.0

# QUESTION 2A
# Hash family: h maps k-grams (strings) → [m], m > 10,000
# h_{a,b}(x) = (a * hash(x) + b) % m

M = 100_003  # prime > 10,000 as the hash range [m]

def make_hash_funcs_v2(t, m=M, seed=42):
    """
    Generate t hash functions of the form:
        h(gram) = (a * hash(gram) + b) % m
    Maps directly from k-gram string -> [m].
    """
    random.seed(seed)
    p = 2**31 - 1  # large Mersenne prime for mixing
    params = [(random.randint(1, p-1), random.randint(0, p-1)) for _ in range(t)]
    return [lambda gram, a=a, b=b: (a * hash(gram) + b) % m for a, b in params]

def minhash_signature(gram_set, hash_funcs):
    """
    For each hash function, find the minimum hash value
    over all grams in the set. This IS the min-hash signature.
    """
    sig = []
    for h in hash_funcs:
        min_val = min(h(gram) for gram in gram_set)
        sig.append(min_val)
    return sig

def approx_jaccard(sig_a, sig_b):

    matches = sum(1 for a, b in zip(sig_a, sig_b) if a == b)
    return matches / len(sig_a)

# Build 3-gram sets for D1 and D2
grams_d1 = char_kgrams(docs['D1'], 3)
grams_d2 = char_kgrams(docs['D2'], 3)

exact_j = jaccard(grams_d1, grams_d2)

print("=" * 60)
print("QUESTION 2A: MIN-HASH APPROXIMATE JACCARD (D1 vs D2)")
print(f"Hash family: h(gram) = (a*hash(gram) + b) % m,  m = {M}")
print(f"Exact Jaccard(D1, D2) [3-char grams] = {exact_j:.6f}")
print("=" * 60)

t_values = [20, 60, 150, 300, 600]

print(f"\n{'t':>6}  {'Approx J':>12}  {'Error':>10}  {'Time (s)':>10}")
print("-" * 46)

results = {}
for t in t_values:

    hash_funcs = make_hash_funcs_v2(t, M)
    start = time.time()
    sig_d1 = minhash_signature(grams_d1, hash_funcs)
    sig_d2 = minhash_signature(grams_d2, hash_funcs)
    elapsed = time.time() - start
    approx = approx_jaccard(sig_d1, sig_d2)
    error = abs(approx - exact_j)
    results[t] = {'approx': approx, 'error': error, 'time': elapsed}
    print(f"  t={t:>4}  {approx:>12.6f}  {error:>10.6f}  {elapsed:>10.4f}")

# QUESTION 2B: Run more experiments to find a good t

print("\n" + "=" * 60)
print("QUESTION 2B: FINDING A GOOD VALUE OF t")
print("Running 10 trials per t-value to measure average error & time")
print("=" * 60)

extra_t_values = [10, 20, 40, 60, 100, 150, 200, 300, 600]
TRIALS = 10

print(f"\n{'t':>6}  {'Avg Approx J':>14}  {'Avg |Error|':>12}  {'Avg Time (s)':>14}")
print("-" * 54)

for t in extra_t_values:

    approx_vals = []
    times = []

    for trial in range(TRIALS):
        hf = make_hash_funcs_v2(t, M, seed=trial * 7 + 13)
        start = time.time()
        s1 = minhash_signature(grams_d1, hf)
        s2 = minhash_signature(grams_d2, hf)
        times.append(time.time() - start)
        approx_vals.append(approx_jaccard(s1, s2))
        
    avg_approx = sum(approx_vals) / TRIALS
    avg_error  = sum(abs(v - exact_j) for v in approx_vals) / TRIALS
    avg_time   = sum(times) / TRIALS
    print(f"  t={t:>4}  {avg_approx:>14.6f}  {avg_error:>12.6f}  {avg_time:>14.4f}")

print(f"""
CONCLUSION (Part B):
────────────────────
Exact Jaccard(D1, D2) = {exact_j:.6f}

""")

QUESTION 2A: MIN-HASH APPROXIMATE JACCARD (D1 vs D2)
Hash family: h(gram) = (a*hash(gram) + b) % m,  m = 100003
Exact Jaccard(D1, D2) [3-char grams] = 0.977979

     t      Approx J       Error    Time (s)
----------------------------------------------
  t=  20      1.000000    0.022021      0.0086
  t=  60      1.000000    0.022021      0.0332
  t= 150      0.960000    0.017979      0.0584
  t= 300      0.973333    0.004646      0.1095
  t= 600      0.976667    0.001313      0.2274

QUESTION 2B: FINDING A GOOD VALUE OF t
Running 10 trials per t-value to measure average error & time

     t    Avg Approx J   Avg |Error|    Avg Time (s)
------------------------------------------------------
  t=  10        0.980000      0.037617          0.0042
  t=  20        0.980000      0.033212          0.0076
  t=  40        0.975000      0.025000          0.0138
  t=  60        0.975000      0.017142          0.0237
  t= 100        0.977000      0.011404          0.0373
  t= 150        0.974667  